# Sistema de Recomendação por Bayesian Personalized Ranking (BPR)

Este notebook implementa um sistema de recomendação baseado em **Fatoração de Matriz com Ranking Personalizado Bayesiano** (BPR — *Bayesian Personalized Ranking*).

## Como o Algoritmo Funciona

### O Problema do SVD para Ranking

O SVD minimiza o **erro quadrático médio** sobre os ratings observados — ele responde à pergunta *"qual nota o usuário daria a este filme?"*. Mas recomendação é um problema de **ranking**: não precisamos saber a nota exata, apenas se o usuário prefere o filme A ao filme B.

O BPR, proposto por Rendle et al. (2009), ataca diretamente o problema de ranking com uma abordagem **pairwise** (pareada): dado um usuário $u$, um item positivo $i$ (que ele avaliou bem) e um item negativo $j$ (que ele não avaliou), aprendemos a ordenar $i$ acima de $j$.

### O Critério BPR-OPT

A função objetivo maximiza a log-verossimilhança de que itens positivos são rankeados acima de itens negativos para cada usuário:

$$\text{BPR-OPT} = \sum_{(u,i,j) \in D_S} \ln \sigma\!\left(\hat{r}_{u,i} - \hat{r}_{u,j}\right) - \lambda_{\theta} \|\theta\|^2$$

Onde:
- $D_S = \{(u, i, j) \mid i \in I_u^+ \text{ e } j \notin I_u^+\}$ — trincas de treino: usuário $u$, item positivo $i$, item negativo $j$
- $\sigma(x) = 1/(1+e^{-x})$ — função sigmoide
- $\hat{r}_{u,i} - \hat{r}_{u,j}$ — diferença de score entre positivo e negativo
- $\lambda_{\theta}$ — coeficiente de regularização L2

### O Modelo de Fatoração de Matriz (BPR-MF)

O score de preferência do usuário $u$ pelo filme $i$ é o mesmo produto interno da MF clássica:

$$\hat{r}_{u,i} = \mathbf{p}_u \cdot \mathbf{q}_i = \sum_{f=1}^{k} p_{u,f} \cdot q_{i,f}$$

Onde:
- $\mathbf{p}_u \in \mathbb{R}^k$ — vetor de fatores latentes do usuário $u$
- $\mathbf{q}_i \in \mathbb{R}^k$ — vetor de fatores latentes do filme $i$
- $k$ — dimensão do espaço latente

A diferença está em **como** $P$ e $Q$ são aprendidas: não via decomposição matricial, mas via **Gradiente Estocástico** (SGD) sobre as trincas $(u, i, j)$.

### Regras de Atualização SGD

As derivadas parciais do BPR-OPT em relação a cada parâmetro geram as atualizações:

$$\mathbf{p}_u \leftarrow \mathbf{p}_u + \alpha \left[(1 - \sigma(x_{uij}))(\mathbf{q}_i - \mathbf{q}_j) - \lambda \mathbf{p}_u\right]$$
$$\mathbf{q}_i \leftarrow \mathbf{q}_i + \alpha \left[(1 - \sigma(x_{uij}))\mathbf{p}_u - \lambda \mathbf{q}_i\right]$$
$$\mathbf{q}_j \leftarrow \mathbf{q}_j + \alpha \left[-(1 - \sigma(x_{uij}))\mathbf{p}_u - \lambda \mathbf{q}_j\right]$$

O fator $(1 - \sigma(x_{uij}))$ é o **gradiente** da log-verossimilhança: ele é grande quando $\hat{r}_{u,i} \approx \hat{r}_{u,j}$ (modelo inseguro da preferência) e pequeno quando $\hat{r}_{u,i} \gg \hat{r}_{u,j}$ (modelo já confiante).

### Definição de Itens Positivos

O BPR foi originalmente proposto para **feedback implícito** (cliques, visualizações). Com ratings explícitos, definimos como positivo qualquer item com `RATING >= RELEVANCE_THRESHOLD` — a mesma definição usada nas métricas de avaliação.

## BPR vs SVD

| Aspecto | MF-SVD | BPR-MF |
|---------|--------|--------|
| Objetivo | Minimiza RMSE dos ratings | Maximiza AUC do ranking |
| Tipo de feedback | Explícito (ratings) | Implícito ou explícito limiarizado |
| Imputação | Requer baseline para NaN | Não precisa de imputação |
| Treinamento | Fatoração exata (ARPACK) | SGD iterativo sobre trincas |
| Score | Predição de rating | Preferência relativa |
| Hiperparâmetros | $k$, método de imputação | $k$, $\alpha$, $\lambda$, épocas |

## Dados Utilizados

| Arquivo | Conteúdo |
|---------|----------|
| `movie_encoding.tsv` | Features dos filmes (gêneros, idiomas, países) |
| `user_ratings.csv` | Avaliações dos usuários (USERID, MOVIEID, RATING) |
| `recommendations.tsv` | Histórico de recomendações anteriores (opcional) |
| `movie_category.tsv` | Categorias de diversidade para métricas de Commonality |

In [ ]:
!pip install numpy pandas scipy matplotlib tqdm --quiet

import numpy, pandas, scipy, matplotlib
print(f'numpy {numpy.__version__} | pandas {pandas.__version__} | scipy {scipy.__version__}')

In [ ]:
import os
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.sparse.linalg import svds
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print('Bibliotecas carregadas com sucesso.')

## 1. Upload dos Arquivos

Monte o Google Drive e configure os caminhos para os arquivos abaixo.

| Arquivo | Descrição | Obrigatório |
|---------|-----------|-------------|
| `movie_encoding.tsv` | Features dos filmes — gêneros, idiomas, países (gerado por `rs_movie_encoding.py`) | Sim |
| `user_ratings.csv` | Avaliações dos usuários: colunas `USERID`, `MOVIEID`, `RATING` | Sim |
| `recommendations.tsv` | Histórico de recomendações passadas: colunas `USERID`, `MOVIEID` | Não |
| `movie_category.tsv` | Categorias de diversidade: `director_women`, `director_nowhite`, etc. (gerado por `rs_movie_category.py`) | Para Commonality |

> **Dica**: coloque todos os arquivos em uma pasta no Google Drive (ex: `MinhaUnidade/rs_data/`) e ajuste `DRIVE_BASE` abaixo.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Configure os caminhos aqui ────────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/rs_data/'

MOVIE_ENC_PATH = os.path.join(DRIVE_BASE, 'movie_encoding.tsv')
RATINGS_PATH   = os.path.join(DRIVE_BASE, 'user_ratings.csv')
RECS_PATH      = os.path.join(DRIVE_BASE, 'recommendations.tsv')   # opcional
CATEGORY_PATH  = os.path.join(DRIVE_BASE, 'movie_category.tsv')    # para Commonality

for label, path, required in [
    ('movie_encoding.tsv', MOVIE_ENC_PATH, True),
    ('user_ratings.csv',   RATINGS_PATH,   True),
    ('recommendations.tsv',RECS_PATH,      False),
    ('movie_category.tsv', CATEGORY_PATH,  False),
]:
    exists = os.path.exists(path)
    status = '✓' if exists else ('✗ OBRIGATÓRIO' if required else '— opcional')
    print(f'  {status}  {label}')

## 2. Carregamento e Exploração dos Dados

Antes de aplicar o SVD, é importante entender a estrutura dos dados:

- **Esparsidade**: diferentemente dos métodos de vizinhança, o SVD lida bem com alta esparsidade pois a imputação baseline preenche os valores ausentes antes da decomposição. Ainda assim, esparsidade muito alta (>99.9%) pode fazer a decomposição convergir para vieses ao invés de padrões de preferência.
- **Distribuição de ratings**: a média global $\mu$ e a variância influenciam diretamente a qualidade da imputação baseline e a convergência do SVD.
- **Tamanho da matriz**: a complexidade do SVD truncado é $O(n_{users} \times n_{items} \times k)$. Para matrizes muito grandes, `scipy.sparse.linalg.svds` usa o método de Krylov (ARPACK) que é eficiente mesmo para matrizes esparsas.

In [ ]:
# ── Carrega os dados ──────────────────────────────────────────────────────────
print('Carregando movie_encoding.tsv ...')
movies_df = pd.read_csv(MOVIE_ENC_PATH, sep='\t', low_memory=False)
genre_cols = [c for c in movies_df.columns if c.startswith('genre_')]
print(f'  {len(movies_df):,} filmes | {len(genre_cols)} gêneros')

print('Carregando user_ratings.csv ...')
ratings_df = pd.read_csv(RATINGS_PATH)
ratings_df.columns = ratings_df.columns.str.upper()
ratings_df = ratings_df.dropna(subset=['USERID','MOVIEID','RATING'])
ratings_df['USERID']  = ratings_df['USERID'].astype(int)
ratings_df['MOVIEID'] = ratings_df['MOVIEID'].astype(int)
ratings_df['RATING']  = ratings_df['RATING'].astype(float)
print(f'  {len(ratings_df):,} avaliações | {ratings_df["USERID"].nunique():,} usuários | {ratings_df["MOVIEID"].nunique():,} filmes')

if os.path.exists(RECS_PATH):
    recs_df = pd.read_csv(RECS_PATH, sep='\t')
    recs_df.columns = recs_df.columns.str.upper()
    print(f'Recomendações anteriores: {len(recs_df):,}')
else:
    recs_df = None

# ── Gráficos exploratórios ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploração dos Dados de Avaliações', fontsize=14, fontweight='bold')

# 1. Distribuição de ratings
ax = axes[0, 0]
ratings_df['RATING'].hist(bins=20, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Distribuição de Ratings')
ax.set_xlabel('Rating')
ax.set_ylabel('Frequência')
ax.axvline(ratings_df['RATING'].mean(), color='crimson', linestyle='--',
           label=f'Média global μ: {ratings_df["RATING"].mean():.2f}')
ax.legend()

# 2. Avaliações por usuário
ax = axes[0, 1]
n_per_user = ratings_df.groupby('USERID').size()
ax.hist(n_per_user, bins=50, color='darkorange', edgecolor='white', log=True)
ax.set_title('Avaliações por Usuário (escala log)')
ax.set_xlabel('Número de filmes avaliados')
ax.set_ylabel('Nº de usuários (log)')
ax.axvline(n_per_user.median(), color='crimson', linestyle='--',
           label=f'Mediana: {n_per_user.median():.0f}')
ax.legend()

# 3. Avaliações por filme
ax = axes[1, 0]
n_per_movie = ratings_df.groupby('MOVIEID').size()
ax.hist(n_per_movie, bins=50, color='seagreen', edgecolor='white', log=True)
ax.set_title('Avaliações por Filme (escala log)')
ax.set_xlabel('Número de avaliações')
ax.set_ylabel('Nº de filmes (log)')
ax.axvline(n_per_movie.median(), color='crimson', linestyle='--',
           label=f'Mediana: {n_per_movie.median():.0f}')
ax.legend()

# 4. Top gêneros
ax = axes[1, 1]
if genre_cols:
    genre_counts = movies_df[genre_cols].sum().sort_values(ascending=False).head(12)
    genre_counts.index = genre_counts.index.str.replace('genre_', '').str.replace('_', ' ').str.title()
    genre_counts.plot(kind='barh', ax=ax, color='mediumpurple')
    ax.set_title('Top-12 Gêneros no Catálogo')
    ax.set_xlabel('Número de filmes')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

n_users_total  = ratings_df['USERID'].nunique()
n_movies_total = ratings_df['MOVIEID'].nunique()
sparsity = 1 - len(ratings_df) / (n_users_total * n_movies_total)
print(f'\nEsparsidade da matriz: {sparsity:.4%}')
print(f'Tamanho da matriz R  : {n_users_total:,} × {n_movies_total:,} = {n_users_total*n_movies_total:,} entradas')

## 3. Divisão Treino / Teste por Usuário (80/20)

Para avaliar o algoritmo, dividimos as avaliações de **cada usuário individualmente** em 80% treino e 20% teste.

No contexto do SVD, essa divisão é importante por uma razão adicional: ao construir a matriz R, usamos **apenas os ratings de treino**. Os ratings de teste permanecem "escondidos" — o SVD não tem acesso a eles, e os usamos para verificar se as predições do modelo são corretas.

> ⚠️ **Atenção ao vazamento de dados (data leakage)**: o SVD é treinado sobre R_treino e a imputação baseline usa médias calculadas **somente sobre R_treino**. Usar dados do teste na imputação seria data leakage e inflaria artificialmente as métricas.

**Parâmetros configuráveis:**
- `TEST_RATIO`: proporção do teste (padrão: 0.2 = 20%).
- `MIN_RATINGS_SPLIT`: usuários com menos que este número ficam inteiramente no treino.
- `RELEVANCE_THRESHOLD`: rating mínimo para considerar um filme como "relevante".

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
TEST_RATIO          = 0.20
MIN_RATINGS_SPLIT   = 5
RELEVANCE_THRESHOLD = 4.0

# ── Split por usuário ─────────────────────────────────────────────────────────
train_ratings: dict[int, dict[int, float]] = defaultdict(dict)
test_ratings:  dict[int, dict[int, float]] = defaultdict(dict)

for uid, group in ratings_df.groupby('USERID'):
    rows = group[['MOVIEID', 'RATING']].values.tolist()
    random.shuffle(rows)

    if len(rows) < MIN_RATINGS_SPLIT:
        for mid, rat in rows:
            train_ratings[uid][int(mid)] = float(rat)
    else:
        n_test = max(1, int(len(rows) * TEST_RATIO))
        for mid, rat in rows[n_test:]:
            train_ratings[uid][int(mid)] = float(rat)
        for mid, rat in rows[:n_test]:
            test_ratings[uid][int(mid)]  = float(rat)

train_ratings = dict(train_ratings)
test_ratings  = dict(test_ratings)

train_users      = [u for u in train_ratings if len(train_ratings[u]) >= MIN_RATINGS_SPLIT]
all_movies_train = sorted(set(m for u in train_ratings.values() for m in u))

n_test_users = sum(1 for u in train_users if u in test_ratings and
                   any(r >= RELEVANCE_THRESHOLD for r in test_ratings[u].values()))

MIN_RATING = float(ratings_df['RATING'].min())
MAX_RATING = float(ratings_df['RATING'].max())

print(f'Usuários totais      : {ratings_df["USERID"].nunique():,}')
print(f'Usuários no treino   : {len(train_users):,}')
print(f'Usuários com teste   : {len(test_ratings):,}')
print(f'  (com item relevante): {n_test_users:,}')
print(f'Filmes no catálogo   : {len(all_movies_train):,}')
print(f'Rating range         : [{MIN_RATING}, {MAX_RATING}]')
print(f'\nRatings no treino    : {sum(len(v) for v in train_ratings.values()):,}')
print(f'Ratings no teste     : {sum(len(v) for v in test_ratings.values()):,}')

## 4. Treinamento BPR-MF via SGD

### Pipeline de treinamento

**Passo 1 — Inicialização** dos fatores latentes com distribuição normal $\mathcal{N}(0, 0.01)$:
$$P \in \mathbb{R}^{n_{users} \times k}, \quad Q \in \mathbb{R}^{n_{items} \times k}$$

**Passo 2 — Definição dos itens positivos** por usuário:
$$I_u^+ = \{i \mid r_{u,i} \geq \theta\} \quad \text{onde } \theta = \texttt{RELEVANCE\_THRESHOLD}$$

**Passo 3 — Por época**, para cada uma das $S$ amostras:
1. Sorteia um usuário $u$ uniformemente de $\{u \mid I_u^+ \neq \emptyset\}$
2. Sorteia item positivo $i \sim \text{Uniform}(I_u^+)$
3. Sorteia item negativo $j \sim \text{Uniform}(I \setminus I_u^{\text{rated}})$ por rejeição
4. Calcula o score diferencial: $x_{uij} = \mathbf{p}_u \cdot (\mathbf{q}_i - \mathbf{q}_j)$
5. Aplica as atualizações SGD com gradiente $(1 - \sigma(x_{uij}))$

**Passo 4 — Reconstrói $\hat{R}$** após o treinamento para extração de scores:
$$\hat{R} = P \cdot Q^\top \quad (n_{users} \times n_{items})$$

### Por que amostrar negativos de itens não avaliados?

Não sabemos se um item não avaliado é realmente negativo — o usuário pode simplesmente não tê-lo descoberto. O BPR assume o pressuposto mais conservador: **itens não avaliados são preferidos menos** do que itens avaliados positivamente, sem afirmar que são ruins.

### Hiperparâmetros

| Parâmetro | Descrição | Valor típico |
|-----------|-----------|-------------|
| `N_FACTORS` | Dimensão do espaço latente $k$ | 20–200 |
| `LEARN_RATE` | Taxa de aprendizado $\alpha$ | 0.01–0.1 |
| `REGULARIZATION` | Penalidade L2 $\lambda$ | 0.0001–0.01 |
| `N_EPOCHS` | Épocas de treinamento | 50–200 |
| `N_SAMPLES` | Amostras por época | $\approx$ total de itens positivos |

> **Critério de convergência**: monitore a curva de loss. Queda inicial rápida seguida de plateau indica convergência. Loss crescente após muitas épocas indica overfitting — reduza `N_EPOCHS` ou aumente `REGULARIZATION`.

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
N_FACTORS        = 50       # dimensão do espaço latente k (mesmo do SVD para comparação)
LEARN_RATE       = 0.05     # taxa de aprendizado α
REGULARIZATION   = 0.001    # regularização L2 λ
N_EPOCHS         = 100      # épocas de treinamento
BPR_POS_THRESHOLD = RELEVANCE_THRESHOLD   # rating >= threshold → item positivo

# ── Índices (mesma estrutura do SVD para compatibilidade com células seguintes) ──
train_users_list = list(train_users)
user_to_idx  = {u: i for i, u in enumerate(train_users_list)}
movie_to_idx = {m: j for j, m in enumerate(all_movies_train)}
n_users_svd  = len(train_users_list)    # nome mantido para compatibilidade
n_movies_svd = len(all_movies_train)

global_mean = float(np.mean([r for movies in train_ratings.values() for r in movies.values()]))

# ── Itens positivos e conjunto de avaliados por usuário ──────────────────────
print(f'Construindo conjuntos positivos (rating ≥ {BPR_POS_THRESHOLD}) ...')
user_pos_idx: dict[int, np.ndarray] = {}
user_rated_set: dict[int, set[int]] = {}

for uid, movies in train_ratings.items():
    rated_idx = {movie_to_idx[m] for m in movies if m in movie_to_idx}
    pos_idx   = np.array(
        [movie_to_idx[m] for m, r in movies.items()
         if m in movie_to_idx and r >= BPR_POS_THRESHOLD],
        dtype=np.int32
    )
    user_rated_set[uid] = rated_idx
    if len(pos_idx) > 0:
        user_pos_idx[uid] = pos_idx

bpr_users       = [u for u in train_users_list if u in user_pos_idx]
n_bpr_users     = len(bpr_users)
total_positives = sum(len(v) for v in user_pos_idx.values())
N_SAMPLES       = total_positives   # 1 época ≈ todos os pares positivos vistos 1×

print(f'  Usuários com itens positivos : {n_bpr_users:,}')
print(f'  Total de itens positivos     : {total_positives:,}')
print(f'  Amostras por época (N_SAMPLES): {N_SAMPLES:,}')

# ── Inicializa fatores latentes ───────────────────────────────────────────────
np.random.seed(42)
P = np.random.normal(0.0, 0.01, (n_users_svd,  N_FACTORS))  # (n_users  × k)
Q = np.random.normal(0.0, 0.01, (n_movies_svd, N_FACTORS))  # (n_movies × k)

print(f'\nIniciando BPR-SGD: {N_EPOCHS} épocas × {N_SAMPLES:,} amostras ...')
print(f'  k={N_FACTORS} | lr={LEARN_RATE} | λ={REGULARIZATION}')

# ── Loop de treinamento BPR-SGD ───────────────────────────────────────────────
epoch_losses   = []
bpr_u_idx_arr  = np.array([user_to_idx[u] for u in bpr_users], dtype=np.int32)

for epoch in range(N_EPOCHS):
    epoch_loss = 0.0

    for _ in range(N_SAMPLES):
        # 1. Amostra usuário
        u_idx = int(bpr_u_idx_arr[np.random.randint(n_bpr_users)])
        uid   = train_users_list[u_idx]

        # 2. Amostra item positivo i
        pos_arr = user_pos_idx[uid]
        i_idx   = int(pos_arr[np.random.randint(len(pos_arr))])

        # 3. Amostra item negativo j por rejeição
        rated = user_rated_set[uid]
        j_idx = int(np.random.randint(n_movies_svd))
        while j_idx in rated:
            j_idx = int(np.random.randint(n_movies_svd))

        # 4. Score diferencial x_uij = p_u · (q_i - q_j)
        diff  = Q[i_idx] - Q[j_idx]
        x_uij = float(P[u_idx].dot(diff))

        # 5. Sigmoid numericamente estável
        x_clip = max(-30.0, min(30.0, x_uij))
        sig    = 1.0 / (1.0 + np.exp(-x_clip))
        g      = 1.0 - sig                      # gradiente de -ln σ(x_uij)

        epoch_loss -= np.log(sig + 1e-10)

        # 6. Gradientes para Q calculados com P_u antes da atualização
        grad_qi =  g * P[u_idx] - REGULARIZATION * Q[i_idx]
        grad_qj = -g * P[u_idx] - REGULARIZATION * Q[j_idx]

        # 7. Atualizações SGD
        P[u_idx] += LEARN_RATE * (g * diff - REGULARIZATION * P[u_idx])
        Q[i_idx] += LEARN_RATE * grad_qi
        Q[j_idx] += LEARN_RATE * grad_qj

    epoch_losses.append(epoch_loss / N_SAMPLES)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'  Época {epoch+1:3d}/{N_EPOCHS} | Loss médio: {epoch_losses[-1]:.5f}')

print(f'\nTreinamento concluído!')
print(f'  Loss inicial  : {epoch_losses[0]:.5f}')
print(f'  Loss final    : {epoch_losses[-1]:.5f}')
print(f'  Redução total : {(1 - epoch_losses[-1]/epoch_losses[0])*100:.1f}%')

# ── Gráfico de convergência inline ───────────────────────────────────────────
fig_conv, ax_conv = plt.subplots(figsize=(8, 4))
ax_conv.plot(range(1, N_EPOCHS + 1), epoch_losses, color='steelblue', linewidth=1.5)
ax_conv.fill_between(range(1, N_EPOCHS + 1), epoch_losses, alpha=0.15, color='steelblue')
ax_conv.set_xlabel('Época')
ax_conv.set_ylabel('Loss médio por amostra (-ln σ)')
ax_conv.set_title(f'Convergência BPR-SGD  (k={N_FACTORS}, lr={LEARN_RATE}, λ={REGULARIZATION})')
ax_conv.axhline(epoch_losses[-1], color='crimson', linestyle='--', alpha=0.7,
                label=f'Loss final: {epoch_losses[-1]:.4f}')
ax_conv.legend()
plt.tight_layout()
plt.show()

## 5. Extração de Scores

Após o treinamento, o score de preferência do usuário $u$ pelo filme $i$ é:

$$\hat{r}_{u,i} = \mathbf{p}_u \cdot \mathbf{q}_i$$

Para eficiência, precomputamos a matriz completa de scores:

$$\hat{R} = P \cdot Q^\top \quad \in \mathbb{R}^{n_{users} \times n_{items}}$$

Assim, a extração de scores por usuário é um simples lookup de linha — idêntico ao SVD.

> **Escala dos scores**: scores BPR são produtos internos de vetores latentes aprendidos, sem escala definida. Para compatibilidade com a célula de ranking (`MIN_SCORE_STOCHASTIC = 3.0`), os scores são normalizados para $[r_{\min}, r_{\max}]$ via min-max global. A normalização preserva a **ordem relativa** dentro de cada usuário.

In [ ]:
# ── Precomputa R̂ = P @ Q.T ───────────────────────────────────────────────────
print('Calculando R̂ = P @ Q.T ...')
R_hat = (P @ Q.T).astype(np.float32)   # (n_users, n_movies)

# Normaliza scores BPR para [MIN_RATING, MAX_RATING] — preserva ordenação
r_min = float(R_hat.min())
r_max = float(R_hat.max())
if r_max > r_min:
    R_hat = (R_hat - r_min) / (r_max - r_min) * (MAX_RATING - MIN_RATING) + MIN_RATING

print(f'  Shape de R̂ : {R_hat.shape}')
print(f'  Score médio : {R_hat.mean():.3f}  (normalizado para [{MIN_RATING}, {MAX_RATING}])')

# ── Extrai scores para filmes não avaliados por cada usuário ─────────────────
user_scores: dict[int, dict[int, float]] = {}

for uid in tqdm(train_users_list, desc='Extraindo scores'):
    i = user_to_idx[uid]

    rated_indices = np.array(
        [movie_to_idx[m] for m in train_ratings[uid] if m in movie_to_idx],
        dtype=np.int32
    )
    mask = np.ones(n_movies_svd, dtype=bool)
    if rated_indices.size > 0:
        mask[rated_indices] = False

    unrated_idx = np.where(mask)[0]
    scores_arr  = R_hat[i, unrated_idx]

    user_scores[uid] = {
        all_movies_train[j]: float(scores_arr[k])
        for k, j in enumerate(unrated_idx)
    }

n_scored            = sum(len(s) for s in user_scores.values())
n_users_with_scores = sum(1 for s in user_scores.values() if s)
all_score_vals      = [s for sc in user_scores.values() for s in sc.values()]

print(f'\nScores extraídos!')
print(f'  Usuários com ao menos 1 score : {n_users_with_scores:,}')
print(f'  Total de (usuário, filme) com score: {n_scored:,}')
print(f'  Score médio : {np.mean(all_score_vals):.3f}')
print(f'  Score mín   : {np.min(all_score_vals):.3f}')
print(f'  Score máx   : {np.max(all_score_vals):.3f}')

## 6. Geração do Ranking de Recomendações

Com os scores calculados, geramos duas estratégias de ranking:

### Top-N Determinístico
Ordena todos os filmes com score pelo valor decrescente e retorna os **N primeiros**. Maximiza a relevância esperada mas produz recomendações estáticas.

### Top-N Estocástico
Converte os scores em **probabilidades** via softmax e amostra N filmes **sem reposição**:

$$p(i) = \frac{e^{\hat{r}_{u,i} / T}}{\sum_{j} e^{\hat{r}_{u,j} / T}}$$

O parâmetro de **temperatura** $T$ controla a aleatoriedade:
- $T \to 0$: equivale a Top-N determinístico.
- $T = 1$: proporcional ao score predito.
- $T \to \infty$: completamente aleatório.

No SVD, scores muito similares entre itens são comuns (o espaço latente é contínuo). O estocástico com temperatura adequada introduz **diversidade** sem perder demasiada qualidade.

**Parâmetros:**
- `RANKING_METHOD`: `'topn'` ou `'stochastic'`
- `N_RECOMMENDATIONS`: número de filmes por recomendação
- `MIN_SCORE_STOCHASTIC`: score mínimo para entrar no sorteio estocástico
- `TEMPERATURE`: temperatura do softmax

In [ ]:
# ── Parâmetros configuráveis ──────────────────────────────────────────────────
RANKING_METHOD       = 'topn'   # 'topn' | 'stochastic'
N_RECOMMENDATIONS    = 10
MIN_SCORE_STOCHASTIC = 3.0
TEMPERATURE          = 1.0

# ── Funções de ranking ────────────────────────────────────────────────────────

def top_n_ranking(scores: dict, n: int, exclude: set) -> list:
    filtered = {m: s for m, s in scores.items() if m not in exclude}
    return sorted(filtered, key=filtered.get, reverse=True)[:n]


def stochastic_ranking(scores: dict, n: int, min_score: float,
                        temperature: float, exclude: set) -> list:
    eligible = {m: s for m, s in scores.items()
                if s >= min_score and m not in exclude}
    if not eligible:
        return top_n_ranking(scores, n, exclude)

    movies = list(eligible.keys())
    raw_s  = np.array([eligible[m] for m in movies], dtype=np.float64)
    logits = raw_s / max(temperature, 1e-8)
    exp_s  = np.exp(logits - logits.max())
    probs  = exp_s / exp_s.sum()

    k = min(n, len(movies))
    chosen = np.random.choice(len(movies), size=k, replace=False, p=probs)
    return [movies[i] for i in chosen]


# ── Gera rankings ─────────────────────────────────────────────────────────────
user_rankings: dict[int, list] = {}

for uid in tqdm(train_users_list, desc='Gerando rankings'):
    scores  = user_scores.get(uid, {})
    exclude = set(train_ratings[uid].keys())

    if RANKING_METHOD == 'stochastic':
        ranking = stochastic_ranking(scores, N_RECOMMENDATIONS,
                                      MIN_SCORE_STOCHASTIC, TEMPERATURE, exclude)
    else:
        ranking = top_n_ranking(scores, N_RECOMMENDATIONS, exclude)

    user_rankings[uid] = ranking

lens = [len(r) for r in user_rankings.values()]
print(f'Rankings gerados: {len(user_rankings):,} usuários')
print(f'  Método        : {RANKING_METHOD.upper()}')
print(f'  Tamanho médio : {np.mean(lens):.1f} filmes')
print(f'  Com ranking   : {sum(1 for l in lens if l > 0):,} usuários')

## 7. Métricas de Avaliação

Avaliamos a qualidade do ranking com as seguintes métricas, implementadas em `rs_metrics.py`:

| Métrica | O que mede | Fórmula resumida |
|---------|------------|------------------|
| **Precision@K** | De K recomendações, quantas são relevantes? | `|top-K ∩ relevantes| / K` |
| **Recall@K** | De todos os relevantes, quantos estão no top-K? | `|top-K ∩ relevantes| / |relevantes|` |
| **F1@K** | Equilíbrio entre Precision e Recall | `2·P·R / (P+R)` |
| **NDCG@K** | Qualidade do ranking ponderando pela posição | Ganho descontado normalizado |
| **MAP** | Precisão média ponderada pela posição do primeiro acerto | Média dos Average Precisions |
| **Hit Rate** | % de usuários com ao menos 1 item relevante | `usuários_com_hit / total` |
| **MRR** | Posição média do primeiro item relevante | `média(1/posição_do_primeiro_hit)` |
| **Coverage** | % do catálogo que o sistema recomenda | `|filmes_únicos_recomendados| / |catálogo|` |

**Itens relevantes**: filmes no conjunto de teste com `RATING >= RELEVANCE_THRESHOLD`.

In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/rs_data/')
from rs_metrics import RankingMetrics

K_METRICS = N_RECOMMENDATIONS

rankings_eval = []
relevant_eval = []
eval_user_ids = []

for uid in train_users_list:
    if uid not in user_rankings or not user_rankings[uid]:
        continue
    if uid not in test_ratings:
        continue

    relevant = [m for m, r in test_ratings[uid].items() if r >= RELEVANCE_THRESHOLD]
    if not relevant:
        continue

    rankings_eval.append(user_rankings[uid])
    relevant_eval.append(relevant)
    eval_user_ids.append(uid)

print(f'Usuários avaliados: {len(eval_user_ids):,}')

m = RankingMetrics()

prec  = np.mean([m.precision_at_k(r, rel, K_METRICS) for r, rel in zip(rankings_eval, relevant_eval)])
rec   = np.mean([m.recall_at_k(r, rel, K_METRICS)    for r, rel in zip(rankings_eval, relevant_eval)])
ndcg  = np.mean([m.ndcg_at_k(r, rel, K_METRICS)      for r, rel in zip(rankings_eval, relevant_eval)])
f1    = m.f1_score(prec, rec)
map_s = m.mean_average_precision(rankings_eval, relevant_eval)
hr    = m.hit_rate(rankings_eval, relevant_eval)
mrr   = m.mean_reciprocal_rank(rankings_eval, relevant_eval)
cov   = m.coverage(rankings_eval, catalog_size=len(all_movies_train))

metric_results = {
    f'Precision@{K_METRICS}': prec,
    f'Recall@{K_METRICS}':    rec,
    f'F1@{K_METRICS}':        f1,
    f'NDCG@{K_METRICS}':      ndcg,
    'MAP':                     map_s,
    'Hit Rate':                hr,
    'MRR':                     mrr,
    'Coverage':                cov,
}

print(f'\n{"Métrica":<20}  {"Valor":>8}')
print('─' * 32)
for name, val in metric_results.items():
    print(f'{name:<20}  {val:>8.4f}')

## 8. Métrica de Commonality (Diversidade de Exposição)

A métrica **Commonality**, implementada em `rs_commonality.py`, mede a probabilidade de que **todos os usuários simultaneamente** se familiarizem com uma categoria de filmes.

### Conceitos

**Probabilidade de Browsing** (RBP — *Rank-Biased Precision*):
$$\Pr(k) = (1 - \gamma) \cdot \gamma^{k-1}$$

**Familiaridade** do usuário $u$ com a categoria $g$:
$$\Pr(F_{u,g} | \pi_u) = \sum_{k=1}^{N} \Pr(k) \cdot R(\pi_u, k, g)$$

**Commonality** para todos os usuários:
$$C_g(\pi) = \prod_{u} \Pr(F_{u,g} | \pi_u)$$

### Interpretação no contexto do SVD

O SVD otimiza a predição de rating sem considerar diversidade. A Commonality permite verificar se o modelo, ao priorizar filmes com alto score predito, acaba reforçando vieses do catálogo: por exemplo, se filmes de diretores homens brancos americanos tendem a ter mais avaliações (e portanto mais influência no SVD), o sistema pode sistematicamente subrecomendado categorias de diversidade.

### Categorias de Diversidade

| Categoria | Critério |
|-----------|----------|
| Diretoras Mulheres | `director_women = 1` |
| Dir. Raça Não-Branca | `director_nowhite = 1` |
| Dir. Região Não-EU/NA | `director_region = 1` |
| Origem Não-EU/NA | `movie_region = 1` |
| Produção Brasileira | `movie_region_bra = 1` |

In [ ]:
from rs_commonality import CommonalityCalculator

GAMMA              = 0.8
COMMONALITY_SAMPLE = 200

if not os.path.exists(CATEGORY_PATH):
    raise FileNotFoundError(
        f'Arquivo não encontrado: {CATEGORY_PATH}\n'
        'Execute rs_movie_category.py para gerar movie_category.tsv.'
    )

cat_df = pd.read_csv(CATEGORY_PATH, sep='\t')
print(f'Categorias carregadas: {len(cat_df):,} filmes')

DIVERSITY_CATEGORIES = {
    'Diretoras Mulheres':     set(cat_df[cat_df['director_women']   == 1]['movieid']),
    'Dir. Raça Não-Branca':   set(cat_df[cat_df['director_nowhite'] == 1]['movieid']),
    'Dir. Reg. Não-EU/NA':    set(cat_df[cat_df['director_region']  == 1]['movieid']),
    'Origem Não-EU/NA':       set(cat_df[cat_df['movie_region']     == 1]['movieid']),
    'Produção Brasileira':    set(cat_df[cat_df['movie_region_bra'] == 1]['movieid']),
}

for cat, items in DIVERSITY_CATEGORIES.items():
    print(f'  {cat:<26}: {len(items):,} filmes')

eligible_users  = [u for u in train_users_list if user_rankings.get(u)]
sample_size     = min(COMMONALITY_SAMPLE, len(eligible_users))
sample_uids     = random.sample(eligible_users, sample_size)
sample_rankings = [user_rankings[u] for u in sample_uids]

print(f'\nAmostra de {sample_size} usuários para Commonality (γ={GAMMA})')

calc = CommonalityCalculator(gamma=GAMMA)

commonality_results = {}
print(f'\n{"Categoria":<26}  {"Fam. Média":>10}  {"Fam. Std":>9}  {"Commonality":>12}')
print('─' * 65)

for cat_name, cat_items in DIVERSITY_CATEGORIES.items():
    cat_list = list(cat_items)

    familiarities = [calc.familiarity(ranking, cat_list) for ranking in sample_rankings]
    commonality   = calc.categoria_commonality(sample_rankings, cat_list)

    commonality_results[cat_name] = {
        'fam_mean':    float(np.mean(familiarities)),
        'fam_std':     float(np.std(familiarities)),
        'commonality': float(commonality),
        'n_items':     len(cat_items),
    }

    fam_m = commonality_results[cat_name]['fam_mean']
    fam_s = commonality_results[cat_name]['fam_std']
    comm  = commonality_results[cat_name]['commonality']
    print(f'{cat_name:<26}  {fam_m:>10.4f}  {fam_s:>9.4f}  {comm:>12.6f}')

print(f'\nNota: Commonality é o produto das familiaridades — decresce com mais usuários.')

## 9. Visualização dos Resultados

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    f'Resultados — BPR-MF  |  Método: {RANKING_METHOD.upper()}  |  '
    f'Top-{N_RECOMMENDATIONS}  |  k={N_FACTORS} fatores latentes',
    fontsize=13, fontweight='bold'
)

# ── 1. Métricas de avaliação ──────────────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
metric_names = list(metric_results.keys())
metric_vals  = list(metric_results.values())
colors_m = ['steelblue' if v > 0.1 else 'lightsteelblue' for v in metric_vals]
bars = ax1.barh(metric_names, metric_vals, color=colors_m, edgecolor='white')
ax1.set_xlim(0, max(metric_vals) * 1.25)
ax1.set_title('Métricas de Avaliação', fontweight='bold')
ax1.set_xlabel('Valor')
for bar, val in zip(bars, metric_vals):
    ax1.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', fontsize=8)
ax1.invert_yaxis()

# ── 2. Familiaridade média por categoria ─────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
cat_names = list(commonality_results.keys())
fam_means = [commonality_results[c]['fam_mean'] for c in cat_names]
fam_stds  = [commonality_results[c]['fam_std']  for c in cat_names]
y_pos = list(range(len(cat_names)))
ax2.barh(y_pos, fam_means, xerr=fam_stds, color='darkorange',
         edgecolor='white', capsize=4, alpha=0.8)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(cat_names, fontsize=9)
ax2.set_title('Familiaridade Média por Categoria\n(com desvio padrão)', fontweight='bold')
ax2.set_xlabel('Familiaridade esperada')
ax2.invert_yaxis()

# ── 3. Commonality por categoria ──────────────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
comm_vals = [commonality_results[c]['commonality'] for c in cat_names]
ax3.barh(y_pos, comm_vals, color='seagreen', edgecolor='white', alpha=0.8)
ax3.set_yticks(y_pos)
ax3.set_yticklabels(cat_names, fontsize=9)
ax3.set_title(f'Commonality por Categoria\n(amostra {sample_size} usuários)', fontweight='bold')
ax3.set_xlabel('Commonality (produto das familiaridades)')
ax3.xaxis.set_major_formatter(mticker.ScalarFormatter(useMathText=True))
ax3.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
ax3.invert_yaxis()

# ── 4. Curva de loss do treinamento BPR ──────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
ax4.plot(range(1, len(epoch_losses) + 1), epoch_losses,
         color='steelblue', linewidth=1.5, label='Loss BPR-SGD')
ax4.fill_between(range(1, len(epoch_losses) + 1), epoch_losses,
                  alpha=0.15, color='steelblue')
ax4.axhline(epoch_losses[-1], color='crimson', linestyle='--', alpha=0.7,
            label=f'Loss final: {epoch_losses[-1]:.4f}')
ax4.set_title(f'Curva de Loss BPR-SGD\n(k={N_FACTORS}, lr={LEARN_RATE}, λ={REGULARIZATION})',
              fontweight='bold')
ax4.set_xlabel('Época')
ax4.set_ylabel('Loss médio por amostra (-ln σ)')
ax4.legend(fontsize=8)

# ── 5. Distribuição dos scores preditos ───────────────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
sample_scores = np.random.choice(all_score_vals, min(50000, len(all_score_vals)), replace=False)
ax5.hist(sample_scores, bins=50, color='mediumpurple', edgecolor='white', alpha=0.8)
ax5.axvline(np.mean(sample_scores), color='crimson', linestyle='--',
            label=f'Média: {np.mean(sample_scores):.2f}')
ax5.axvline(global_mean, color='orange', linestyle=':',
            label=f'Média global: {global_mean:.2f}')
ax5.axvline(MIN_SCORE_STOCHASTIC, color='green', linestyle=':',
            label=f'Min estocástico: {MIN_SCORE_STOCHASTIC}')
ax5.set_title('Distribuição de Scores Preditos', fontweight='bold')
ax5.set_xlabel('Score predito $\\hat{r}_{u,i}$')
ax5.set_ylabel('Frequência')
ax5.legend(fontsize=7)

# ── 6. Tamanho das categorias de diversidade ──────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
n_items_cat   = [commonality_results[c]['n_items'] for c in cat_names]
total_catalog = len(all_movies_train)
pcts = [v / total_catalog * 100 if total_catalog > 0 else 0 for v in n_items_cat]
ax6.barh(y_pos, pcts, color='steelblue', edgecolor='white', alpha=0.8)
ax6.set_yticks(y_pos)
ax6.set_yticklabels(cat_names, fontsize=9)
ax6.set_title(f'Tamanho das Categorias\n(% do catálogo — {total_catalog:,} filmes)', fontweight='bold')
ax6.set_xlabel('% do catálogo')
for i, (pct, n) in enumerate(zip(pcts, n_items_cat)):
    ax6.text(pct + 0.3, i, f'{n:,} ({pct:.1f}%)', va='center', fontsize=8)
ax6.invert_yaxis()

plt.tight_layout()
plt.savefig('resultados_bpr.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo em resultados_bpr.png')